# FATES Satellite Phenology (FATES-SP) simulations at NEON sites

<div class="alert alert-info">
<strong>Exercise: Use the CTSM workflow to run a single point FATES-sp case at a NEON site</strong><br><br>

Create and run a case called **`BONA.FATESsp`** using the:
- compset ``I1PtClm60FatesRsGs``, the 
- resolution ``CLM_USRDAT``, and that 
- applies ``USERMODS`` to set up a NEON site.

The `run_tower` script we've been using is very useful, but it simplifies a number of steps that users may want to take control of in their workflow.  We'll introduce how you can control aspects of your case setup and configuration here. For example, you can use the CTSM workflow to configure your case to run a FATES case at a supported tower site.  We'll use material from the basic tutorial and customizing your case for this example
</div>

## Background

For a Satellite Phenology (SP) case, we want to run NEON sites for a bit to let soil water and temperatures equilibrate before comparing to observations. Luckily, this only takes a few years (in most cases).

### Usermods
A **usermod directory** (“user modifications” directory) is a bundle of case-specific settings that CLM/CIME can apply when you create or clone a case. It’s a convenient way to start from a known configuration rather than manually setting every option afterward. The usermods may include files such as CLM namelist settings (`user_nl_clm`), settings for other model components, and XML configuration changes. 

You pass the bundle to CIME when creating a case. For example:
```shell
./create_newcase ... --user-mods-dir clm/NEON/FATES/BONA
```

Have a look at the standard usermods we provide in CLM6 in the `cime_config` directory of your CTSM checkout:
```shell
ls /glade/u/home/$USER/code/my_clm_code/cime_config/usermods_dirs/clm
```

<div class="alert alert-info">
<strong>Exercise </strong><br>

- Move into the NEON usermods directory:

```
cd /glade/u/home/$USER/code/my_clm_code/cime_config/usermods_dirs/clm/NEON
```

- List its contents:
```
ls -1
```

You should see several directories, including:
```
ABBY
BARR
BART
BLAN
BONA
CLBJ
CPER
DCFS
defaults
DEJU
DELA
DSNY
FATES
GRSM
...
```

- You'll notice that FATES has its own directory.  Have a look inside.
```
ls FATES
```

- The `default` directory in NEON/FATES contains shell commands and `user_nl_*` settings that are shared across all sites. Look and see.
```
ls -1 FATES/defaults
```

expected output includes:
```
shell_commands
user_nl_clm
user_nl_cpl
user_nl_datm
user_nl_datm_streams
```
You can look to see what default settings your NEON cases will inherit. 

- Each NEON site also includes custom XML changes.  Look and see:
```
ls -1 FATES/BONA
```

expected output includes:
```
include_user_mods
shell_commands
```

**Check Your Understanding**

How are NEON site-level details passed to CLM? 

*Hint: look at the `shell_commands` in the CLM `usermods_dirs` for NEON defaults and sites*

</div>

## Get Started

Let's create a single-point FATES Satellite Phenology (FATES-SP) case at a NEON site.


<div class="alert alert-warning">  

**Single Point FATES-SP Overview**
 
**How do you customize your case to run a FATES-SP experiment?**

Specifically you'll want to make namelist changes that:
- set up a FATES-SP case.
  - **NOTE** These should be done with a compset in CLM6, but we don't have a single-point FATES-SP compset at this time.
- modify history file output for an SP case to get some daily data to look at

And XML changes to run:
- From a 'cold start' (as we don't have initial conditions);
  - see spinup lecture, and [users guide](https://escomp.github.io/CTSM/users_guide/running-special-cases/Spinning-up-the-biogeochemistry-Bgc-spinup.html) 
- A quick spinup to initialize the soil moisture state (4 years). This isn't long enough, but OK for purposes of the tutorial;
- For the full observational record.
</div>



<div class="alert alert-success">   
<details>
<summary><font face="Times New Roman" color='blue' size=+1>Click here for the solution</font></summary><br>
       
**Create a new case <font face="Courier" color='purple'><strong>BONA.FATESsp </strong></font>**

```shell
cd /glade/u/home/$USER/code/my_clm_code/cime/scripts
```

Change the 4-character NEON site below:

```shell
neon_site="BONA"
```

Then create a new case.
```shell
./create_newcase --case /glade/u/home/$USER/cases/${neon_site}.FATESsp \
    --res CLM_USRDAT \
    --compset I1PtClm60FatesRsGs \
    --run-unsupported \
    --user-mods-dir clm/NEON/FATES/${neon_site} \
    --handle-preexisting-dirs r

```

**Case <font face="Courier" color='purple'><strong>setup</strong></font>**

```shell
cd /glade/u/home/$USER/cases/BONA.FATESsp
./case.setup
```

**Case <font face="Courier" color='purple'><strong>customization</strong></font>**

**user_nl_clm** changes.  We currently don't have a single-point FATES-SP compset in CLM.  We're working on it, but in the meantime you can request satellite phenology in the single-point compset we chose above by adding the following into `user_nl_clm`.
```shell
echo "use_fates_sp = .true." >> user_nl_clm
```

The NEON FATES usermods already add a second history file (`h1`) to your case. Have a look:

```shell
grep -A3 hist_ user_nl_clm
```

You'll see a variable list (`hist_fincl2`) set up to write **every model time step** (`hist_nhtfrq(2) = 1`), with 48 time steps per file (`hist_mfilt(2) = 48`). That's a lot of files for an 8-year run! We'll keep the same variables, but switch to **daily averages**, with one year per file:

```shell
echo "hist_nhtfrq(2) = -24" >> user_nl_clm
echo "hist_mfilt(2)  = 365" >> user_nl_clm
echo "hist_fincl2 = 'FCEV', 'FCTR', 'FGEV', 'FIRA', 'FSA', 'FSH', 'FATES_GPP', 'FATES_GPP_PF', 'H2OSOI', 'SNOW_DEPTH', 'TBOT', 'TSOI'" >> user_nl_clm
```

A negative `hist_nhtfrq` value is a number of hours, so `-24` means daily. Because the fields are averaged, these files will be named `*.clm2.h1a.*`.

If you set the same variable twice in `user_nl_clm`, **the last entry wins**, so these lines override the usermods settings.

You'll learn more about history settings in the namelist modifications exercise.

**XML changes**
<br><br>
First look and see what default settings are for:
- Running from a cold start (without initial conditions)
- What years of DATM data do we have?
- How long are we running the case?

```shell
./xmlquery CLM_FORCE_COLDSTART
./xmlquery DATM_YR_START DATM_YR_ALIGN DATM_YR_END
./xmlquery STOP_OPTION STOP_N 
```

Expected results are:
```
CLM_FORCE_COLDSTART: off

DATM_YR_START: 2018
DATM_YR_ALIGN: 2018
DATM_YR_END: 2021

STOP_OPTION: ndays
STOP_N: 5
```

For most NEON sites we have at least 4 complete years of data to work with (2018-2021). 
Now we'll change this to:
- Force a cold start
- Cycle over the full 4 years of DATM data available two times (once for spinup, and once for actual comparison to observations)
- Start the simulation in 2014

```shell
./xmlchange CLM_FORCE_COLDSTART=on
./xmlchange STOP_OPTION=nyears
./xmlchange STOP_N=8  
./xmlchange RUN_STARTDATE=2014-01-01 
```

**<font face="Courier" color='purple'><strong>Preview namelist</strong></font>**

Run `./preview_namelists` to ensure these changes were applied correctly.

Check that your `lnd_in` file looks OK, e.g.:
- `use_fates_sp = .true.`
- `fsurdat` should be one with 16pfts in the name
- `finidat = ' '`
- etc...

```shell
grep use_fates_sp CaseDocs/lnd_in
grep fsurdat CaseDocs/lnd_in
grep finidat CaseDocs/lnd_in
grep hist_fincl2 CaseDocs/lnd_in
grep hist_mfilt CaseDocs/lnd_in
```

**Case <font face="Courier" color='purple'><strong>build</strong></font>**

```shell
qcmd -- ./case.build 
```

**Case <font face="Courier" color='purple'><strong>submit</strong></font>**

```shell
./case.submit
```

Check the status:

```shell
qstat -u $USER
```    

When the run is completed, look into the archive directory for: 
<font face="Courier" color='purple'><strong>BONA.FATESsp</strong></font>.  
    
Check that your archive directory on derecho (the path will be different on other machines): 

```shell
cd /glade/derecho/scratch/$USER/archive/BONA.FATESsp
ls 
```

</details>
</div>

## Test your understanding

- Did your run complete successfully?
- Where is your case directory?
- Where is your run directory?
- What sorts of files did you get as output?
- What years did you run over?